### Import Dependencies

In [2]:
import joblib
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)


### Loading Processed Data

In [3]:
data_file_path = "../data/processed_data.csv"
data_matrix = pd.read_csv(data_file_path)

X = data_matrix.drop(columns=["Churn"])
y = data_matrix["Churn"]

### Splitting the Data in Train and Test Data

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25, stratify=y, random_state=2210)

### Training Potential Models

In [5]:
models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    
    "Logistic Regression": make_pipeline(
        StandardScaler(), 
        LogisticRegression(max_iter=5000,class_weight ='balanced'), 
        
    ),
  

    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'),
    
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)
}


param_grids = {
    "Baseline": {},
    
    "Logistic Regression": {
        'logisticregression__C': [0.1, 1, 10],
        'logisticregression__solver': ['lbfgs', 'liblinear']
    },
    
    
    "Random Forest": {
        'n_estimators': [100, 200],
        'max_depth': [None, 10],
        'min_samples_split': [2, 5]
    },
    
    "XGBoost": {
        'n_estimators': [100, 200,1000],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 6]
    }
}

### Evaluation of the first models

In [6]:
best_models = {}
test_scores = {}

for name, model in models.items():
    print(f"\n--- Tuning & Training: {name} ---")
    
    if param_grids[name]:
        grid_search = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            cv=5,
            scoring="f1",  # Recomandat 'f1' sau 'roc_auc' pentru Churn nebalansat
            n_jobs=-1
        )
        grid_search.fit(X_train, y_train)
        
        best_models[name] = grid_search.best_estimator_
        print(f"Best Params: {grid_search.best_params_}")
    else:
        model.fit(X_train, y_train)
        best_models[name] = model

   
    y_pred = best_models[name].predict(X_test)
    score = f1_score(y_test, y_pred)
    test_scores[name] = score
    print(f"F1 score on test data: {score:.4f}")


best_model_name = max(test_scores, key=test_scores.get)
print(f"\n The best model is : {best_model_name} (F1 Score: {test_scores[best_model_name]:.4f})")


--- Tuning & Training: Baseline ---
F1 score on test data: 0.0000

--- Tuning & Training: Logistic Regression ---
Best Params: {'logisticregression__C': 0.1, 'logisticregression__solver': 'lbfgs'}
F1 score on test data: 0.6247

--- Tuning & Training: Random Forest ---
Best Params: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}
F1 score on test data: 0.6282

--- Tuning & Training: XGBoost ---
Best Params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
F1 score on test data: 0.6017

 The best model is : Random Forest (F1 Score: 0.6282)


### Save the best model on Disk

In [7]:

joblib.dump(best_models[best_model_name], '../models/best_churn_model.pkl')
print(" The best model was successfully saved in the folder named models/")

 The best model was successfully saved in the folder named models/
